In [8]:
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 3.1 MB/s eta 0:00:00a 0:00:01


In [27]:
import sys
import csv
import json
import praw
import yaml
import typer
from datetime import datetime

# noinspection PyUnresolvedReferences
import pretty_errors  # keep the import to have better error messages

from os.path import join
from pathlib import Path
from typer import Argument
from typer import Option
from typing import Optional, List
from loguru import logger
from codetiming import Timer
from pushshift_py import PushshiftAPI
from prawcore.exceptions import NotFound

In [11]:
user_agent = "u/DevDevReddit"
reddit = praw.Reddit(
        client_id="2P4KSmTyeg-XFmjaGn9kow",
        client_secret="ladizOud9Hz8oboPqlCOS611t8PMgQ",
        user_agent=user_agent,
    )

In [24]:
headlines = set ( )
reddit.subreddit('')
for submission in reddit.subreddit('LearnToReddit').new(limit  = 10) :
    print("Check Submiission")
    print (submission.title)
    for comment in submission.comments:
        print(comment.body)
    #print(submission.selftext)
    #print(submission.full_text)
    #print (submission.id)
    #print (submission.author) 
    #print (submission.created_utc)
    #print (submission.score)
    #print (submission.upvote_ratio)
    #print (submission.url)  
    #break
    headlines. add (submission.title)
print (len (headlines) )

Check Submiission
Testing Link Posts
Welcome to r/LearnToReddit, /u/DevDevReddit! Thanks for posting. Someone will be along to help you with your post shortly.

If you forgot to describe what you are posting to test in your post title, please let us know here in comments so we can let you know how you did.

If your post isn't a practice post of some kind, you may be looking for another community. There is r/newtoreddit and r/help for help redditing, or see r/findareddit for help finding the right place. 

Thank you! :)  


*I am a bot, and this action was performed automatically. Please [contact the moderators of this subreddit](/message/compose/?to=/r/LearnToReddit) if you have any questions or concerns.*
Check Submiission
Testing embedded video
Welcome to r/LearnToReddit, /u/Inner_thoughts_loud! Thanks for posting. Someone will be along to help you with your post shortly.

If you forgot to describe what you are posting to test in your post title, please let us know here in comments s

In [34]:
class OutputManager:
    """
    Class used to collect and store data (submissions and comments)
    """
    params_filename = "params.yaml"

    def __init__(self, output_dir: str, subreddit: str):
        self.submissions_list = []
        self.submissions_raw_list = []
        self.comments_list = []
        self.comments_raw_list = []
        self.run_id = datetime.today().strftime('%Y%m%d%H%M%S')

        self.subreddit_dir = join(output_dir, subreddit)
        self.runtime_dir = join(self.subreddit_dir, self.run_id)

        self.submissions_output = join(self.runtime_dir, "submissions")
        self.sub_raw_output = join(self.runtime_dir, "submissions", "raw")
        self.comments_output = join(self.runtime_dir, "comments")
        self.comments_raw_output = join(self.runtime_dir, "comments", "raw")
        self.params_path = join(self.runtime_dir, OutputManager.params_filename)

        self.total_submissions_counter = 0
        self.total_comments_counter = 0

        for path in [self.submissions_output,
                     self.sub_raw_output,
                     self.comments_output,
                     self.comments_raw_output]:
            Path(path).mkdir(parents=True, exist_ok=True)

    def reset_lists(self):
        self.submissions_list = []
        self.submissions_raw_list = []
        self.comments_list = []
        self.comments_raw_list = []

    def store(self, lap: str):
        # Track total data statistics
        self.total_submissions_counter += len(self.submissions_list)
        self.total_comments_counter += len(self.comments_list)

        # Store the collected data
        dictlist_to_csv(join(self.submissions_output, f"{lap}.csv"), self.submissions_list)
        dictlist_to_csv(join(self.comments_output, f"{lap}.csv"), self.comments_list)

        if len(self.submissions_raw_list) > 0:
            with open(join(self.sub_raw_output, f"{lap}.njson"), "a", encoding="utf-8") as f:
                f.write("\n".join(json.dumps(row) for row in self.submissions_raw_list))
        if len(self.comments_raw_list) > 0:
            with open(join(self.comments_raw_output, f"{lap}.njson"), "a", encoding="utf-8") as f:
                f.write("\n".join(json.dumps(row, default=lambda o: '<not serializable>')
                                  for row in self.comments_raw_list))

    def store_params(self, params: dict):
        with open(self.params_path, "w", encoding="utf-8") as f:
            yaml.dump(params, f)

    def load_params(self) -> dict:
        with open(self.params_path, "r", encoding="utf-8") as f:
            params = yaml.load(f, yaml.FullLoader)
        return params

    def enrich_and_store_params(self, utc_older: int, utc_newer: int):
        params = self.load_params()
        params["utc_older"] = utc_older
        params["utc_newer"] = utc_newer
        params["total_comments_counter"] = self.total_comments_counter
        params["total_submissions_counter"] = self.total_submissions_counter
        params["total_counter"] = self.total_comments_counter + self.total_submissions_counter
        self.store_params(params)


def dictlist_to_csv(file_path: str, dictionaries_list: List[dict]):
    if len(dictionaries_list) == 0:
        dictionaries_list = [{}]
    keys = dictionaries_list[0].keys()
    with open(file_path, 'w', newline='', encoding="utf-8") as output_file:
        dict_writer = csv.DictWriter(output_file, keys, dialect="excel")
        dict_writer.writeheader()
        dict_writer.writerows(dictionaries_list)


def init_locals(debug: str,
                output_dir: str,
                subreddit: str,
                utc_upper_bound: str,
                utc_lower_bound: str,
                run_args: dict,
                ) -> (str, OutputManager):
    assert not (utc_upper_bound and utc_lower_bound), "`utc_lower_bound` and " \
                                                      "`utc_upper_bound` parameters are in mutual exclusion"
    run_args.pop("reddit_secret")

    if not debug:
        logger.remove()
        logger.add(sys.stderr, level="INFO")

    direction = "after" if utc_upper_bound else "before"
    output_manager = OutputManager(output_dir, subreddit)

    output_manager.store_params(run_args)
    return direction, output_manager


def init_clients(reddit_id: str,
                 reddit_secret: str,
                 reddit_username: str
                 ) -> (PushshiftAPI, praw.Reddit):
    pushshift_api = PushshiftAPI()

    reddit_api = praw.Reddit(
        client_id=reddit_id,
        client_secret=reddit_secret,
        user_agent=f"python_script:subreddit_downloader:(by /u/{reddit_username})",
    )

    return pushshift_api, reddit_api


def utc_range_calculator(utc_received: int,
                         utc_upper_bound: int,
                         utc_lower_bound: int
                         ) -> (int, int):
    """
    Calculate the max UTC range seen.

    Increase/decrease utc_upper_bound/utc_lower_bound according with utc_received value
    """
    if not utc_upper_bound or not utc_lower_bound:
        utc_upper_bound = utc_received
        utc_lower_bound = utc_received

    utc_lower_bound = utc_lower_bound if utc_received > utc_lower_bound else utc_received
    utc_upper_bound = utc_upper_bound if utc_received < utc_upper_bound else utc_received

    return utc_lower_bound, utc_upper_bound


def comments_fetcher(sub, output_manager, reddit_api, comments_cap):
    """
    Comments fetcher
    Get all comments with depth-first approach
    Solution from https://praw.readthedocs.io/en/latest/tutorials/comments.html
    """
    try:
        submission_rich_data = reddit_api.submission(id=sub.id)
        logger.debug(f"Requesting {submission_rich_data.num_comments} comments...")
        submission_rich_data.comments.replace_more(limit=comments_cap)
        comments = submission_rich_data.comments.list()
    except NotFound:
        logger.warning(f"Submission not found in PRAW: `{sub.id}` - `{sub.title}` - `{sub.full_link}`")
        return
    for comment in comments:
        comment_useful_data = {
            "id": comment.id,
            "submission_id": sub.id,
            "body": comment.body.replace('\n', '\\n'),
            "created_utc": int(comment.created_utc),
            "parent_id": comment.parent_id,
            "permalink": comment.permalink,
        }
        output_manager.comments_raw_list.append(comment.__dict__)
        output_manager.comments_list.append(comment_useful_data)


def submission_fetcher(sub, output_manager: OutputManager):
    """
    Get and store reddit submission info
    """
    # Sometimes the submission doesn't have the selftext
    self_text_normalized = sub.selftext.replace('\n', '\\n') if hasattr(sub, "selftext") else "<not selftext available>"

    submission_useful_data = {
        "id": sub.id,
        "created_utc": sub.created_utc,
        "title": sub.title.replace('\n', '\\n'),
        "selftext": self_text_normalized,
        "full_link": sub.full_link,
    }
    output_manager.submissions_list.append(submission_useful_data)
    output_manager.submissions_raw_list.append(sub.d_)


class HelpMessages:
    help_reddit_url = "https://github.com/reddit-archive/reddit/wiki/OAuth2"
    help_reddit_agent_url = "https://github.com/reddit-archive/reddit/wiki/API"
    help_praw_replace_more_url = "https://asyncpraw.readthedocs.io/en/latest/code_overview/other/commentforest.html#asyncpraw.models.comment_forest.CommentForest.replace_more"

    subreddit = "The subreddit name"
    output_dir = "Optional output directory"
    batch_size = "Request `batch_size` submission per time"
    laps = "How many times request `batch_size` reddit submissions"
    reddit_id = f"Reddit client_id, visit {help_reddit_url}"
    reddit_secret = f"Reddit client_secret, visit {help_reddit_url}"
    reddit_username = f"Reddit username, used for build the `user_agent` string, visit {help_reddit_agent_url}"
    utc_after = "Fetch the submissions after this UTC date"
    utc_before = "Fetch the submissions before this UTC date"
    debug = "Enable debug logging"
    comments_cap = f"Some submissions have 10k> nested comments and stuck the praw API call." \
                   f"If provided, the system requires new comments `comments_cap` times to the praw API." \
                   f"`comments_cap` under the hood will be passed directly to `replace_more` function as " \
                   f"`limit` parameter. For more info see the README and visit {help_praw_replace_more_url}."
